# 04. Closures, decorators and context managers

**Goal.** Wrap behaviour around functions and resources without touching their code, and know what each wrapper costs.

**You will be able to:**
1. Explain closures through cell objects and avoid the late-binding trap.
2. Write decorators that keep metadata, take arguments and hold state.
3. Write context managers as classes and with `contextlib`.
4. Measure the per-call overhead that wrappers add.

## 1. Closures and cells

### How it works under the hood
- Python resolves names in the order Local, Enclosing, Global, Built-in (LEGB).
- When an inner function uses a variable from an outer function, the compiler stores it in a **cell** object. Both functions share the cell, and it outlives the outer call.
- The inner function keeps its cells in `__closure__`. Variable names are in `__code__.co_freevars`.
- A closure looks up the variable, not its value at creation time. That causes the late-binding trap in loops.
- Assigning to an enclosing name needs `nonlocal`. Without it Python creates a new local.

### Production use
- Factories that build configured callables: rate limiters, validators, route handlers with captured settings.
- Callbacks in GUIs and async code that carry request context.

### Trade-offs

| Tool | Memory | Latency | Readability | Advantages | Disadvantages |
|---|---|---|---|---|---|
| Closure | One cell per captured name plus a function object | Cell read is as fast as a local | Compact for small state | No class boilerplate | State is hidden, hard to inspect or pickle |
| Class with `__call__` | Instance with `__dict__` | Attribute lookup per access | Explicit state and methods | Easy to test and extend | More code |
| `functools.partial` | Small object | C-level call | Very clear | Pre-binds arguments only | No private state |

In [1]:
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "understanding_python").is_dir())
sys.path.insert(0, str(ROOT / "src"))
print("python", sys.version.split()[0], "| package root", ROOT.name)

python 3.13.6 | package root understanding-python


In [2]:
from understanding_python.functional import make_counter


def rate_limiter(max_calls):
    calls = 0

    def allow():
        nonlocal calls
        calls += 1
        return calls <= max_calls

    return allow


limit = rate_limiter(2)
print([limit() for _ in range(4)])
print("free variables:", limit.__code__.co_freevars)
print("cell contents :", [c.cell_contents for c in limit.__closure__])

a, b = make_counter(), make_counter(100)
print("independent cells:", a(), a(), b(), a())

late = [lambda: i for i in range(3)]
print("late binding :", [f() for f in late])
bound = [lambda i=i: i for i in range(3)]
print("default bound:", [f() for f in bound])

[True, True, False, False]
free variables: ('calls', 'max_calls')
cell contents : [4, 2]
independent cells: 1 2 101 3
late binding : [2, 2, 2]
default bound: [0, 1, 2]


## 2. Decorators

### How it works under the hood
- `@d` above `def f` runs `f = d(f)` once, at definition time. The name `f` now points to whatever `d` returned, usually a wrapper closure.
- Without `functools.wraps`, the wrapper hides `__name__`, `__doc__` and the signature. `wraps` copies them and sets `__wrapped__` to the original.
- A decorator factory is one more level: `@retry(attempts=3)` calls `retry(3)` first, then applies the result.
- Stacked decorators apply bottom up and run top down: the outermost wrapper executes first.
- `functools.cache` and `lru_cache` are C-implemented wrappers with a dict keyed by arguments, so arguments must be hashable.

### Production use
- Cross-cutting concerns: retries with back-off, timing, authentication checks, caching, rate limits, structured logging.
- Frameworks register handlers with decorators (`@app.get("/orders")`).
- `@functools.cache` on pure, expensive lookups such as parsing a config file or computing a tax table.

### Trade-offs

| Technique | Memory | Latency | Readability | Advantages | Disadvantages |
|---|---|---|---|---|---|
| Function decorator | One closure per use | Extra Python call per invocation (about 50 to 100 ns) | Declarative at the call site | Reuse without editing the function | Stack traces gain frames, hides signatures without `wraps` |
| Class decorator or `__call__` object | Instance per use | Slower than closure | Clear state | Counters, caches with methods | Does not bind as a method without `__get__` |
| `functools.cache` | Unbounded dict | Dict lookup replaces the work | One line | Huge win on repeated work | Memory leak if keys grow, stale data |
| Inline code | None | None | Repeated boilerplate | No indirection | Duplicated logic |

In [3]:
import functools
import timeit

from understanding_python.functional import retry


def audit(func):
    """Record every call in `func.calls`. Keeps the metadata of the original."""
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        wrapper.calls.append((args, kwargs))
        return func(*args, **kwargs)

    wrapper.calls = []
    return wrapper


@audit
def price_with_vat(net, rate=0.2):
    """Return the gross price."""
    return round(net * (1 + rate), 2)


print(price_with_vat(100), price_with_vat(50, rate=0.05))
print("name and doc kept :", price_with_vat.__name__, "|", price_with_vat.__doc__)
print("original reachable:", price_with_vat.__wrapped__(10))
print("recorded calls    :", price_with_vat.calls)

attempts = []


@retry(attempts=3, exceptions=(ConnectionError,))
def fetch_rates():
    attempts.append(1)
    if len(attempts) < 3:
        raise ConnectionError(f"upstream timeout (try {len(attempts)})")
    return {"GBP": 1.0, "EUR": 1.17}


print("retry result:", fetch_rates(), "after", len(attempts), "tries")


@retry(attempts=2, exceptions=(ConnectionError,))
def always_down():
    raise ConnectionError("503")


try:
    always_down()
except ConnectionError as exc:
    print("gives up and re-raises:", exc)


def trace(label):
    def deco(func):
        @functools.wraps(func)
        def wrapper(*a, **k):
            print(f"  enter {label}")
            out = func(*a, **k)
            print(f"  leave {label}")
            return out
        return wrapper
    return deco


@trace("outer")
@trace("inner")
def work():
    print("  body")


work()


@functools.cache
def tax_band(income):
    sum(range(20_000))  # stands in for a slow table lookup
    return "basic" if income < 50_270 else "higher"


tax_band(30_000)
tax_band(30_000)
tax_band(80_000)
print("cache_info:", tax_band.cache_info())


def plain(x):
    return x + 1


wrapped_light = functools.wraps(plain)(lambda x: plain(x))
t_plain = timeit.timeit(lambda: plain(1), number=500_000)
t_light = timeit.timeit(lambda: wrapped_light(1), number=500_000)
print(f"direct call {t_plain * 2000:.0f} ns | through one wrapper {t_light * 2000:.0f} ns | overhead {(t_light - t_plain) * 2000:.0f} ns")

120.0 52.5
name and doc kept : price_with_vat | Return the gross price.
original reachable: 12.0
recorded calls    : [((100,), {}), ((50,), {'rate': 0.05})]
retry result: {'GBP': 1.0, 'EUR': 1.17} after 3 tries
gives up and re-raises: 503
  enter outer
  enter inner
  body
  leave inner
  leave outer
cache_info: CacheInfo(hits=1, misses=2, maxsize=None, currsize=2)
direct call 93 ns | through one wrapper 130 ns | overhead 37 ns


## 3. Context managers

### How it works under the hood
- `with expr as name:` calls `expr.__enter__()`, binds its result, runs the body, then always calls `__exit__(exc_type, exc, tb)`. It behaves like `try`/`finally`.
- If the body raised, `__exit__` receives the exception. Returning a true value swallows it. Returning false lets it propagate.
- `@contextlib.contextmanager` turns a generator into a manager: code before `yield` is `__enter__`, code after is `__exit__`. Wrap the `yield` in `try`/`finally` so cleanup runs on errors.
- `contextlib.ExitStack` holds a dynamic number of managers and unwinds them in reverse order.

### Production use
- Database transactions: commit on success, roll back on error.
- Files, sockets, locks, temporary directories, patched settings in tests.
- Timing and tracing spans in observability code.

### Trade-offs

| Style | Memory | Latency | Readability | Advantages | Disadvantages |
|---|---|---|---|---|---|
| Class with `__enter__`/`__exit__` | Instance | Two method calls | Explicit, reusable, testable | Full control over exceptions | More lines |
| `@contextmanager` generator | Generator frame | Slightly slower | Short and linear | Fast to write | Forgetting `try`/`finally` skips cleanup |
| `try`/`finally` inline | None | None | Repeated at each site | No abstraction | Easy to forget or duplicate |
| `ExitStack` | Stack of callbacks | Small | Handles variable counts | Dynamic resources | Slightly indirect |

In [4]:
import contextlib
import tempfile
import time
from pathlib import Path

from understanding_python.functional import Suppress, Transaction, timer

ledger = {"alice": 120, "bob": 40}
with Transaction(ledger) as tx:
    tx["alice"] -= 50
    tx["bob"] += 50
print("committed :", ledger)

try:
    with Transaction(ledger) as tx:
        tx["alice"] -= 500
        if tx["alice"] < 0:
            raise ValueError("insufficient funds")
except ValueError as exc:
    print("rolled back:", ledger, "| reason:", exc)

stats = {}
with timer(stats):
    time.sleep(0.05)
print(f"timer recorded {stats['elapsed'] * 1000:.0f} ms")

with Suppress(KeyError):
    ledger["carol"]
print("KeyError swallowed; others still propagate:", end=" ")
try:
    with Suppress(KeyError):
        raise TypeError("bad type")
except TypeError as exc:
    print(exc)


@contextlib.contextmanager
def opened_all(paths, registry):
    """Open every file for writing; close all of them even when one open fails."""
    with contextlib.ExitStack() as stack:
        for p in paths:
            registry.append(stack.enter_context(open(p, "w", encoding="utf8")))
        yield registry


with tempfile.TemporaryDirectory() as tmp:
    names = [Path(tmp) / f"shard-{i}.log" for i in range(3)]
    handles = []
    with opened_all(names, handles):
        for h in handles:
            h.write("ok\n")
    print("files closed after the block:", all(h.closed for h in handles))
    partial = []
    try:
        with opened_all([names[0], Path(tmp) / "missing" / "x.log"], partial):
            pass
    except FileNotFoundError:
        print("second open failed; first handle closed:", partial[0].closed)

with contextlib.suppress(FileNotFoundError):
    Path("does-not-exist.txt").read_text()
print("suppress() used for an optional file")

committed : {'alice': 70, 'bob': 90}
rolled back: {'alice': 70, 'bob': 90} | reason: insufficient funds


timer recorded 50 ms
KeyError swallowed; others still propagate: bad type


files closed after the block: True
second open failed; first handle closed: True
suppress() used for an optional file
